# Autonomous Systems Portfolio 2  
***StarGunner met Reinforced Learning***

|Name     |Studentnumber|Github    |
|---------|-------------|----------|
|Henry Lau|22122958     |HenryLau08|
|Michal Reszka-Gniecki|23025069|ckires|
|Mohamed Belaachir|22143572|mobelaachir|

In [ ]:
import gymnasium as gym
import ale_py
import math
import numpy as np
import random
import matplotlib
import matplotlib.pyplot as plt
from collections import namedtuple, deque
from itertools import count

import tensorflow as tf

gym.register_envs(ale_py)

In [48]:
# set up matplotlib
is_ipython = 'inline' in matplotlib.get_backend()
if is_ipython:
    from IPython import display

plt.ion()

# if GPU is to be used
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

In [49]:
Transition = namedtuple('Transition',
                        ('state', 'action', 'next_state', 'reward'))


class ReplayMemory(object):

    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)

    def push(self, *args):
        """Save a transition"""
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

In [65]:
env_id = 'StarGunner-v4'
env = gym.make(env_id, obs_type="ram", full_action_space=True, mode=0)

In [66]:
observation_space = env.observation_space
state_size = observation_space.shape[0]
print(f'State space: {state_size}')

action_size = env.action_space.n
print(f'Action space: {action_size}')

State space: 128
Action space: 18


In [ ]:
def preprocess_ram(state):
    state = np.array(state, dtype=np.float32) / 255.0
    return torch.tensor(state).unsqueeze(0)

In [ ]:
class DQN:
    def __init__(self, state_size, action_size, learning_rate, name="DQN"):
        self.state_size = state_size
        self.action_size = action_size
        self.learning_rate = learning_rate

        with tf.variable_scope(name):
            self.inputs_ = tf.placeholder(tf.float32, [None, 128], name="inputs")
            self.actions_ = tf.placeholder(tf.float32, [None, self.action_size], name="actions_")
            self.target_Q = tf.placeholder(tf.float32, [None], name="target")

            self.fc1 = tf.layers.dense(self.inputs_, 256, activation=tf.nn.elu, name="fc1")
            self.fc2 = tf.layers.dense(self.fc1, 256, activation=tf.nn.elu, name="fc2")

            self.output = tf.layers.dense(self.fc2, self.action_size)

            # Q(s,a)
            self.Q = tf.reduce_sum(self.output * self.actions_, axis=1)

            # loss
            self.loss = tf.reduce_mean(tf.square(self.target_Q - self.Q))

            # optimizer
            self.optimizer = tf.train.AdamOptimizer(self.learning_rate).minimize(self.loss)


In [ ]:
policy_net = DQN(state_size, action_size, learning_rate=0.1, name="policy")
target_net = DQN(state_size, action_size, learning_rate=0.1, name="target")

In [ ]:
gamma = 0.99
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995
batch_size = 64

In [ ]:
# Q_policy = policy_net.output
# Q_next = target_net.output

# y = r + gamma * max(Q_next)

# optimizer.minimize(loss on policy_net)

# target_net.weights = policy_net.weights

In [ ]:
Transition = namedtuple('Transition',
                        ('state', 'action', 'next_state', 'reward', 'done'))

class ReplayMemory(object):

    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)

    def push(self, *args):
        """Save a transition"""
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)